## Setup and Imports

In [ ]:
import sys
import os
import json
import tempfile
from pathlib import Path

# Add the src directory to Python path
sys.path.append('../src')

from data_pipeline.crawl_data import get_keys, fetch_image
from utils.utils import key_to_url

## Explore Real Data Structure

In [ ]:
# Check available data files
data_dir = Path('../data_test')
print("Available data files:")
for file in data_dir.glob('*.json'):
    print(f"  {file.name}")
    
print("\nAvailable folders:")
for folder in data_dir.iterdir():
    if folder.is_dir():
        print(f"  {folder.name}/")

In [ ]:
# Examine sample data from fashion.json
fashion_file = data_dir / 'fashion.json'

print("First 3 lines of fashion.json:")
with open(fashion_file, 'r') as f:
    for i, line in enumerate(f):
        if i >= 3:
            break
        data = json.loads(line.strip())
        print(f"Line {i+1}: {data}")
        
print("\nData structure:")
with open(fashion_file, 'r') as f:
    sample = json.loads(f.readline().strip())
    for key, value in sample.items():
        print(f"  {key}: {type(value).__name__} = {value}")

## Test get_keys Function

In [ ]:
# Test with fashion data
print("Testing get_keys with fashion.json (first 10 lines):")
keys = get_keys(fashion_file, 10)

print(f"Total unique keys found: {len(keys)}")
print("\nFirst 10 keys:")
for i, key in enumerate(sorted(keys)[:10]):
    print(f"  {i+1}: {key}")
    
# Analyze key characteristics
print("\nKey characteristics:")
key_lengths = [len(key) for key in keys]
print(f"  Key lengths: min={min(key_lengths)}, max={max(key_lengths)}, avg={sum(key_lengths)/len(key_lengths):.1f}")

# Check if all keys are hex
hex_keys = [key for key in keys if all(c in '0123456789abcdefABCDEF' for c in key)]
print(f"  Hex format keys: {len(hex_keys)}/{len(keys)} ({len(hex_keys)/len(keys)*100:.1f}%)")

In [ ]:
# Test with different datasets
datasets = ['fashion.json']

for dataset in datasets:
    filepath = data_dir / dataset
    if os.path.exists(filepath):
        keys = get_keys(filepath, 5)
        print(f"{dataset}: {len(keys)} unique keys from 5 lines")
    else:
        print(f"{dataset}: File not found")

## Test key_to_url Function

In [ ]:
# Test URL generation with real keys
sample_keys = list(sorted(keys)[:5])  # Use first 5 keys from previous test

print("Pinterest URLs generated from real keys:")
for i, key in enumerate(sample_keys):
    try:
        url = key_to_url(key)
        print(f"  {i+1}: {key} -> {url}")
    except Exception as e:
        print(f"  {i+1}: {key} -> ERROR: {e}")

In [ ]:
# Test edge cases for key_to_url
test_cases = [
    ("abcdef123456", "Valid hex key"),
    ("123abc", "Minimum length hex"),
    ("abc", "Too short - should fail"),
    ("ghijkl123456", "Invalid hex - should fail"),
    ("", "Empty key - should fail")
]

print("Testing key_to_url edge cases:")
for key, description in test_cases:
    try:
        url = key_to_url(key)
        print(f"  ✓ {description}: {key} -> {url}")
    except Exception as e:
        print(f"  ✗ {description}: {key} -> ERROR: {e}")

## Test fetch_image Function (Safe Mode)

In [ ]:
# Download real images using real keys from Pinterest
from IPython.display import Image, display
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# Get some real keys from our data
test_keys = list(sorted(keys)[:3])  # Use first 3 keys from previous test
print(f"Downloading REAL images with {len(test_keys)} keys:")

# Create a temporary directory for downloads
with tempfile.TemporaryDirectory() as temp_dir:
    print(f"Temporary directory: {temp_dir}")
    
    # Download real images
    downloaded_files = []
    failed_downloads = []
    
    for i, key in enumerate(test_keys):
        print(f"\n{i+1}. Downloading key: {key}")
        print(f"   URL: {key_to_url(key)}")
        
        try:
            result = fetch_image(key, temp_dir, 0.5, 3)  # 0.5s sleep, max 3 retries
            
            if result:
                file_path = os.path.join(temp_dir, f"{key}.jpg")
                downloaded_files.append((key, file_path))
                print(f"   ✓ Downloaded successfully to {file_path}")
                
                # Get file size
                file_size = os.path.getsize(file_path)
                print(f"   File size: {file_size:,} bytes ({file_size/1024:.1f} KB)")
            else:
                failed_downloads.append(key)
                print("   ✗ Download failed or file already exists")
                
        except Exception as e:
            failed_downloads.append(key)
            print(f"   ✗ Error: {e}")
    
    # Display results summary
    print(f"\n{'='*60}")
    print("DOWNLOAD SUMMARY")
    print(f"{'='*60}")
    print(f"Successful downloads: {len(downloaded_files)}")
    print(f"Failed downloads: {len(failed_downloads)}")
    
    if failed_downloads:
        print(f"Failed keys: {failed_downloads}")
    
    # Display the downloaded images
    if downloaded_files:
        print(f"\n{'='*60}")
        print("DISPLAYING DOWNLOADED IMAGES")
        print(f"{'='*60}")
        
        # Also display using IPython.display.Image for individual images with better quality
        print("\nIndividual image displays:")
        for i, (key, file_path) in enumerate(downloaded_files):
            print(f"\nImage {i+1} - Key: {key}")
            try:
                display(Image(filename=file_path, width=300, height=300))
            except Exception as e:
                print(f"Error displaying image: {e}")
    else:
        print("\n⚠️  No images were successfully downloaded.")
        print("This could be due to:")
        print("- Network connectivity issues")
        print("- Pinterest blocking requests")
        print("- Invalid keys in the dataset")
        print("- Images no longer available at those URLs")

print(f"\n{'='*60}")
print("REAL IMAGE DOWNLOAD COMPLETE")
print(f"{'='*60}")
print("Note: These are actual images downloaded from Pinterest!")
print("If downloads failed, try running the cell again or check your internet connection.")

## Experiment with Command Line Arguments

In [ ]:
# Simulate command line usage
print("Sample command line usage:")
print("\nBasic usage:")
print("python crawl_data.py --input_file ../data/fashion.json --output_dir ./output --max_lines 10")

print("\nWith custom settings:")
print("python crawl_data.py \\")
print("  --input_file ../data/fashion.json \\")
print("  --output_dir ./images \\")
print("  --max_lines 50 \\")
print("  --sleep_time 0.5 \\")
print("  --batch_size 20 \\")
print("  --batch_sleep 2.0 \\")
print("  --max_retries 3 \\")
print("  --log_level DEBUG")

print("\nArgument explanations:")
args_info = {
    "--input_file": "Path to the JSON file containing product/scene data",
    "--output_dir": "Directory where downloaded images will be saved", 
    "--max_lines": "Maximum number of lines to process from input file",
    "--sleep_time": "Base sleep time between individual downloads (seconds)",
    "--batch_size": "Number of images to download before batch sleep",
    "--batch_sleep": "Sleep time between batches (seconds)",
    "--max_retries": "Maximum retry attempts for failed downloads",
    "--log_level": "Logging verbosity (DEBUG, INFO, WARNING, ERROR)"
}

for arg, description in args_info.items():
    print(f"  {arg}: {description}")